# Trustworthy Network Anomaly Detection Lab

This notebook is a reproducible walkthrough. It uses **synthetic labelled telemetry** and should not be interpreted as a claim about operational enterprise performance.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('..').resolve() / 'src'))
from nadlab.data import TelemetryConfig, anomaly_counts, chronological_split, generate_synthetic_telemetry
from nadlab.preprocessing import fit_normal_scaler, make_windows, transform_frame


## 1. Generate controlled telemetry
The generator injects DDoS, port-scan, brute-force, and exfiltration-shaped perturbations with labels and severity.

In [ ]:
frame = generate_synthetic_telemetry(TelemetryConfig(n_hosts=8, n_steps=900, seed=42))
print(frame.shape)
print('Anomaly rate:', f"{frame.label.mean():.2%}")
anomaly_counts(frame)


## 2. Use chronological splits
Scaling is fitted only on normal rows from the training period. This avoids using future observations or anomaly labels to shape the feature transformation.

In [ ]:
train, validation, test = chronological_split(frame)
scaler = fit_normal_scaler(train)
test_windows = make_windows(transform_frame(test, scaler), window_size=24)
test_windows.x.shape, test_windows.y.mean()


## 3. Run a detector benchmark
Use the terminal command below for a full reproducible run. Deep models may take longer on CPU.

```bash
python scripts/run_experiment.py --model all
python scripts/plot_results.py --results results
```